# 🔍 Notebook 11: Hallucination Detection, Claim Decomposition & Evidence-Based Feedback Loop
### Capstone: VL-JEPA vs. CLIP for CCTV Surveillance Event Retrieval, Event Summarization & Hallucination Analysis
---
**Objective:** Evaluate and mitigate LLM hallucination in forensic surveillance reporting:
- **Atomic Claim Decomposition:** Breaking free-text incident narratives into discrete verifiable factual claims.
- **Visual Factuality Verification:** Auditing claims against retrieved keyframes and timestamp metadata.
- **Evidence-Based Feedback Loop:** Automatically flagging unsupported claims, re-querying the visual index, and generating a revised, fact-checked report.
- **Quantitative Metrics:** Hallucination Rate, Claim Precision, and Grounding Faithfulness Score.


## Step 1: Atomic Claim Decomposition Algorithm
Parse generated incident reports into testable propositions (entities, actions, objects, timestamps).


In [1]:
import pandas as pd
import numpy as np

def decompose_claims(report_text: str):
    """Extracts individual claims from report sentences."""
    claims = []
    lines = [line.strip() for line in report_text.split('\n') if line.strip()]
    claim_id = 1
    for line in lines:
        if line.startswith(('-', '*', '1.', '2.', '3.')):
            cleaned = line.lstrip('-* 0123456789.').strip()
            claims.append({
                "claim_id": f"C{claim_id:02d}",
                "claim_text": cleaned,
                "verified": None,
                "confidence": None
            })
            claim_id += 1
    return claims

sample_report_claims = """
- An individual in black clothing entered the convenience store at 00:00:02.
- The suspect pointed a silver revolver at the cashier.
- The cashier opened the cash drawer and handed over banknotes.
- The suspect fled on a red motorcycle waiting outside.
"""

claims_list = decompose_claims(sample_report_claims)
df_claims = pd.DataFrame(claims_list)
print("=== Decomposed Atomic Claims ===")
display(df_claims)


=== Decomposed Atomic Claims ===


,claim_id,claim_text,verified,confidence
0,C01,An individual in black clothing entered the co...,None,None
1,C02,The suspect pointed a silver revolver at the c...,None,None
2,C03,The cashier opened the cash drawer and handed ...,None,None
3,C04,The suspect fled on a red motorcycle waiting o...,None,None


## Step 2: Factuality Verification & Hallucination Flagging
Evaluate each claim against retrieved video keyframes:
- **Supported:** Claim matches verified keyframe evidence.
- **Hallucinated:** Claim asserts an ungrounded entity (e.g., invented weapon type or getaway vehicle).


In [2]:
# Simulate verification results against verified keyframes
audit_results = [
    {"claim_id": "C01", "claim_text": "An individual in black clothing entered the store at 00:00:02", "status": "SUPPORTED", "confidence": 0.94, "evidence_source": "Frame 5 (00:00:02)"},
    {"claim_id": "C02", "claim_text": "The suspect pointed a silver revolver at the cashier", "status": "HALLUCINATED", "confidence": 0.32, "evidence_source": "Frame 15 shows hand gesture, no clear weapon visible"},
    {"claim_id": "C03", "claim_text": "The cashier opened the cash drawer and handed over banknotes", "status": "SUPPORTED", "confidence": 0.88, "evidence_source": "Frame 28 (00:00:08)"},
    {"claim_id": "C04", "claim_text": "The suspect fled on a red motorcycle waiting outside", "status": "HALLUCINATED", "confidence": 0.15, "evidence_source": "Camera view ends at doorway; exterior not captured"}
]

df_audit = pd.DataFrame(audit_results)
total_claims = len(df_audit)
supported = sum(1 for a in audit_results if a["status"] == "SUPPORTED")
hallucinated = sum(1 for a in audit_results if a["status"] == "HALLUCINATED")

hallucination_rate = (hallucinated / total_claims) * 100
claim_precision = (supported / total_claims) * 100

print("=== Factuality Audit Results ===")
display(df_audit)
print(f"\nTotal Claims Evaluated:  {total_claims}")
print(f"Supported Claims:        {supported} ({claim_precision:.1f}% Claim Precision)")
print(f"Hallucinated Claims:     {hallucinated} ({hallucination_rate:.1f}% Hallucination Rate)")


=== Factuality Audit Results ===


,claim_id,claim_text,status,confidence,evidence_source
0,C01,An individual in black clothing entered the st...,SUPPORTED,0.94,Frame 5 (00:00:02)
1,C02,The suspect pointed a silver revolver at the c...,HALLUCINATED,0.32,"Frame 15 shows hand gesture, no clear weapon v..."
2,C03,The cashier opened the cash drawer and handed ...,SUPPORTED,0.88,Frame 28 (00:00:08)
3,C04,The suspect fled on a red motorcycle waiting o...,HALLUCINATED,0.15,Camera view ends at doorway; exterior not capt...



Total Claims Evaluated:  4
Supported Claims:        2 (50.0% Claim Precision)
Hallucinated Claims:     2 (50.0% Hallucination Rate)


## Step 3: Evidence-Based Feedback Loop & Report Revision
Remove hallucinated claims and trigger report regeneration constrained strictly to supported facts.


In [3]:
def regenerate_fact_checked_report(audit_table):
    supported_claims = audit_table[audit_table["status"] == "SUPPORTED"]["claim_text"].tolist()
    flagged_claims = audit_table[audit_table["status"] == "HALLUCINATED"]["claim_text"].tolist()
    
    revised = "[REVISED FORENSIC REPORT - EVIDENCE AUDITED]\n\n"
    revised += "CONFIRMED TIMELINE & ACTIONS:\n"
    for i, c in enumerate(supported_claims, 1):
        revised += f"{i}. {c} [VERIFIED]\n"
        
    revised += "\nEXCLUDED UNVERIFIED ALLEGATIONS (HALLUCINATION SUPPRESSED):\n"
    for f in flagged_claims:
        revised += f"- {f} [REMOVED: Insufficient visual resolution]\n"
    return revised

revised_output = regenerate_fact_checked_report(df_audit)
print(revised_output)


[REVISED FORENSIC REPORT - EVIDENCE AUDITED]

CONFIRMED TIMELINE & ACTIONS:
1. An individual in black clothing entered the store at 00:00:02 [VERIFIED]
2. The cashier opened the cash drawer and handed over banknotes [VERIFIED]

EXCLUDED UNVERIFIED ALLEGATIONS (HALLUCINATION SUPPRESSED):
- The suspect pointed a silver revolver at the cashier [REMOVED: Insufficient visual resolution]
- The suspect fled on a red motorcycle waiting outside [REMOVED: Insufficient visual resolution]

